In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import uuid

np.random.seed(42)
random.seed(42)

In [2]:
NUM_TRANSACTIONS = 500000

BANKS = ['SBI', 'HDFC', 'ICICI', 'Axis', 'Kotak', 'BOB', 'PNB', 'YesBank']
BANK_WEIGHTS = [0.18, 0.08, 0.09, 0.11, 0.07, 0.16, 0.19, 0.12]

# True underlying failure tendency per bank (this is HIDDEN from model)
BANK_BASE_FAIL = {
    'SBI': 0.22, 'HDFC': 0.08, 'ICICI': 0.09, 'Axis': 0.11,
    'Kotak': 0.07, 'BOB': 0.20, 'PNB': 0.24, 'YesBank': 0.18
}

DEVICE_TYPES = ['android', 'ios', 'feature_phone']
DEVICE_WEIGHTS = [0.70, 0.20, 0.10]

NETWORK_TYPES = ['4G', '3G', '2G', 'wifi']
NETWORK_WEIGHTS = [0.50, 0.20, 0.10, 0.20]

FAILURE_CODES = ['RB', 'Z9', 'ZM', 'Z6', 'U30', 'U69', 'XD', 'B1', 'U16', 'RP']
FAILURE_WEIGHTS = [0.35, 0.25, 0.10, 0.10, 0.07, 0.05, 0.04, 0.02, 0.01, 0.01]

In [3]:
def get_upi_id(bank):
    phone = ''.join([str(random.randint(0,9)) for _ in range(10)])
    suffixes = {
        'SBI': 'oksbi', 'HDFC': 'okhdfc', 'ICICI': 'okicici',
        'Axis': 'okaxis', 'Kotak': 'kotak', 'BOB': 'okbob',
        'PNB': 'okpnb', 'YesBank': 'yesbank'
    }
    return f"{phone}@{suffixes[bank]}"

def get_amount():
    amount = np.random.lognormal(mean=6.5, sigma=1.2)
    return round(float(np.clip(amount, 1, 100000)), 2)

def get_amount_bucket(amount):
    if amount < 100: return 'micro'
    elif amount < 1000: return 'small'
    elif amount < 10000: return 'medium'
    elif amount < 100000: return 'large'
    else: return 'very_large'

In [4]:
def generate_transaction():

    # --- timestamp ---
    days_ago = random.randint(0, 180)
    # Peak hour weighting
    hours = list(range(24))
    hour_weights = [
        1.0,1.0,1.0,1.0,1.0,1.0,  # 0-5 night
        1.2,1.5,1.8,               # 6-8 morning
        2.0,2.0,2.0,               # 9-11 peak
        1.8,1.6,1.5,               # 12-14 afternoon
        1.6,1.7,1.8,               # 15-17 evening build
        2.2,2.5,2.5,2.3,           # 18-21 peak evening
        1.8,1.4                    # 22-23
    ]
    hour = random.choices(hours, weights=hour_weights, k=1)[0]
    timestamp = datetime.now() - timedelta(days=days_ago)
    timestamp = timestamp.replace(
        hour=hour,
        minute=random.randint(0,59),
        second=random.randint(0,59)
    )

    day_of_week = timestamp.weekday()
    day_of_month = timestamp.day
    is_salary_day = 1 if day_of_month in [1, 2, 30, 31] else 0
    is_festival_day = 1 if (timestamp.month == 11 and timestamp.day in range(1,6)) else 0

    # --- banks ---
    sender_bank = random.choices(BANKS, weights=BANK_WEIGHTS, k=1)[0]
    receiver_bank = random.choices(BANKS, k=1)[0]

    # --- device & network ---
    device_type = random.choices(DEVICE_TYPES, weights=DEVICE_WEIGHTS, k=1)[0]
    if device_type == 'feature_phone':
        network_type = random.choices(['2G','3G'], weights=[0.7,0.3], k=1)[0]
    else:
        network_type = random.choices(NETWORK_TYPES, weights=NETWORK_WEIGHTS, k=1)[0]

    # --- amount ---
    amount = get_amount()
    amount_bucket = get_amount_bucket(amount)

    # ── KEY FIX: bank_health varies independently per transaction ──
    # Base health from bank tendency + time stress + random variation
    sender_stress = (
        BANK_BASE_FAIL[sender_bank]
        + (0.10 if (19 <= hour <= 22) else 0.04 if (9 <= hour <= 11) else 0)
        + (0.08 if is_salary_day else 0)
        + (0.12 if is_festival_day else 0)
        + np.random.normal(0, 0.06)   # ← independent noise per transaction
    )
    sender_bank_health = round(float(np.clip(1 - sender_stress, 0.15, 0.95)), 2)

    receiver_stress = (
        BANK_BASE_FAIL[receiver_bank]
        + (0.10 if (19 <= hour <= 22) else 0.04 if (9 <= hour <= 11) else 0)
        + (0.08 if is_salary_day else 0)
        + (0.12 if is_festival_day else 0)
        + np.random.normal(0, 0.06)   # ← independent noise per transaction
    )
    receiver_bank_health = round(float(np.clip(1 - receiver_stress, 0.15, 0.95)), 2)

    # ── KEY FIX: recent_fail_rate is fully independent from bank_health ──
    # Sampled from Beta distribution centered on bank's true rate
    s_alpha = max(BANK_BASE_FAIL[sender_bank] * 8, 0.5)
    s_beta  = max((1 - BANK_BASE_FAIL[sender_bank]) * 8, 0.5)
    sender_recent_fail_rate = round(float(np.clip(
        np.random.beta(s_alpha, s_beta) + np.random.normal(0, 0.04),
        0, 1
    )), 2)

    r_alpha = max(BANK_BASE_FAIL[receiver_bank] * 8, 0.5)
    r_beta  = max((1 - BANK_BASE_FAIL[receiver_bank]) * 8, 0.5)
    receiver_recent_fail_rate = round(float(np.clip(
        np.random.beta(r_alpha, r_beta) + np.random.normal(0, 0.04),
        0, 1
    )), 2)

    # ── KEY FIX: failure probability uses STRONG weights ──
    failure_prob = 0.03  # base

    # These three drive 80% of the signal
    failure_prob += (1 - sender_bank_health) * 0.45
    failure_prob += sender_recent_fail_rate   * 0.30
    failure_prob += (1 - receiver_bank_health) * 0.15

    # Secondary signals
    network_boost = {'2G': 0.15, '3G': 0.08, '4G': 0.01, 'wifi': 0.00}
    failure_prob += network_boost[network_type]

    if is_festival_day: failure_prob += 0.12
    if is_salary_day:   failure_prob += 0.08
    if 19 <= hour <= 22 or 9 <= hour <= 11: failure_prob += 0.06
    if amount > 10000:  failure_prob += 0.06
    if device_type == 'feature_phone': failure_prob += 0.04

    failure_prob = float(np.clip(failure_prob, 0, 0.95))

    is_failed = 1 if random.random() < failure_prob else 0
    failure_reason_code = random.choices(
        FAILURE_CODES, weights=FAILURE_WEIGHTS, k=1)[0] if is_failed else None

    return {
        'transaction_id':            str(uuid.uuid4()),
        'timestamp':                 timestamp.strftime('%Y-%m-%d %H:%M:%S'),
        'sender_vpa':                get_upi_id(sender_bank),
        'receiver_vpa':              get_upi_id(receiver_bank),
        'sender_bank':               sender_bank,
        'receiver_bank':             receiver_bank,
        'sender_bank_encoded':       BANKS.index(sender_bank),
        'receiver_bank_encoded':     BANKS.index(receiver_bank),
        'hour_of_day':               hour,
        'day_of_week':               day_of_week,
        'is_salary_day':             is_salary_day,
        'is_festival_day':           is_festival_day,
        'amount':                    amount,
        'amount_bucket':             amount_bucket,
        'device_type':               device_type,
        'network_type':              network_type,
        'sender_bank_health':        sender_bank_health,
        'receiver_bank_health':      receiver_bank_health,
        'sender_recent_fail_rate':   sender_recent_fail_rate,
        'receiver_recent_fail_rate': receiver_recent_fail_rate,
        'failure_reason_code':       failure_reason_code,
        'is_failed':                 is_failed
    }

In [5]:
def generate_dataset(num_transactions=NUM_TRANSACTIONS):
    
    print(f"Generating {num_transactions:,} transactions...")
    
    transactions = []
    
    for i in range(num_transactions):
        transactions.append(generate_transaction())
        
        if (i + 1) % 50000 == 0:
            print(f"  {i + 1:,} transactions generated...")
    
    print("Creating DataFrame...")
    df = pd.DataFrame(transactions)
    
    print("Saving to CSV...")
    df.to_csv('upi_transactions.csv', index=False)
    
    print(f"\nDone! Dataset saved as 'upi_transactions.csv'")
    return df

In [6]:
df = generate_dataset()

print("\n--- Dataset Summary ---")
print(f"Total transactions:  {len(df):,}")
print(f"Total columns:       {len(df.columns)}")
print(f"\nFailure rate:        {df['is_failed'].mean()*100:.1f}%")
print(f"Total failures:      {df['is_failed'].sum():,}")
print(f"Total successes:     {(df['is_failed']==0).sum():,}")

print("\n--- Failure rate by bank ---")
bank_stats = df.groupby('sender_bank')['is_failed'].mean()*100
print(bank_stats.round(1).sort_values(ascending=False))

print("\n--- Failure rate by network ---")
network_stats = df.groupby('network_type')['is_failed'].mean()*100
print(network_stats.round(1).sort_values(ascending=False))

print("\n--- Failure rate by hour (peak hours) ---")
hour_stats = df.groupby('hour_of_day')['is_failed'].mean()*100
print(hour_stats.round(1).sort_values(ascending=False).head(5))

print("\n--- Amount distribution ---")
print(df['amount'].describe().round(2))

print("\n--- Failure reason distribution ---")
print(df[df['is_failed']==1]['failure_reason_code'].value_counts())

print("\n--- First 3 rows ---")
df.head(3)

Generating 500,000 transactions...
  50,000 transactions generated...
  100,000 transactions generated...
  150,000 transactions generated...
  200,000 transactions generated...
  250,000 transactions generated...
  300,000 transactions generated...
  350,000 transactions generated...
  400,000 transactions generated...
  450,000 transactions generated...
  500,000 transactions generated...
Creating DataFrame...
Saving to CSV...

Done! Dataset saved as 'upi_transactions.csv'

--- Dataset Summary ---
Total transactions:  500,000
Total columns:       22

Failure rate:        29.3%
Total failures:      146,350
Total successes:     353,650

--- Failure rate by bank ---
sender_bank
PNB        33.9
SBI        32.9
BOB        31.6
YesBank    30.1
Axis       24.8
ICICI      23.5
HDFC       23.1
Kotak      22.4
Name: is_failed, dtype: float64

--- Failure rate by network ---
network_type
2G      40.8
3G      33.1
4G      25.4
wifi    24.3
Name: is_failed, dtype: float64

--- Failure rate by hou

,transaction_id,timestamp,sender_vpa,receiver_vpa,sender_bank,receiver_bank,sender_bank_encoded,receiver_bank_encoded,hour_of_day,day_of_week,...,amount,amount_bucket,device_type,network_type,sender_bank_health,receiver_bank_health,sender_recent_fail_rate,receiver_recent_fail_rate,failure_reason_code,is_failed
0,c38dea2f-ee4a-4af0-a9b1-ab89cc65c8b4,2025-10-27 04:47:17,9600133890@okhdfc,8386379402@okhdfc,HDFC,HDFC,1,1,4,0,...,1207.20,medium,android,2G,0.93,0.88,0.02,0.05,None,0
1,60bd302b-1fba-46f1-9e94-2f93ec1289f3,2025-10-12 12:17:09,5940781618@okhdfc,4959310341@okpnb,HDFC,PNB,1,6,12,6,...,1275.47,medium,android,4G,0.95,0.79,0.02,0.19,None,0
2,19469d89-681f-4b8a-8173-bdad7eb09b2c,2026-02-08 21:24:17,1928327648@okaxis,3503056413@okpnb,Axis,PNB,3,6,21,6,...,1362.76,medium,android,4G,0.64,0.64,0.04,0.11,None,0


In [7]:
df = pd.read_csv("D:/Jupyter Lab/Fintech Projects/UPI Payment Failure Predictor/data/upi_transactions.csv")
df

,transaction_id,timestamp,sender_vpa,receiver_vpa,sender_bank,receiver_bank,sender_bank_encoded,receiver_bank_encoded,hour_of_day,day_of_week,...,amount,amount_bucket,device_type,network_type,sender_bank_health,receiver_bank_health,sender_recent_fail_rate,receiver_recent_fail_rate,failure_reason_code,is_failed
0,c38dea2f-ee4a-4af0-a9b1-ab89cc65c8b4,2025-10-27 04:47:17,9600133890@okhdfc,8386379402@okhdfc,HDFC,HDFC,1,1,4,0,...,1207.20,medium,android,2G,0.93,0.88,0.02,0.05,NaN,0
1,60bd302b-1fba-46f1-9e94-2f93ec1289f3,2025-10-12 12:17:09,5940781618@okhdfc,4959310341@okpnb,HDFC,PNB,1,6,12,6,...,1275.47,medium,android,4G,0.95,0.79,0.02,0.19,NaN,0
2,19469d89-681f-4b8a-8173-bdad7eb09b2c,2026-02-08 21:24:17,1928327648@okaxis,3503056413@okpnb,Axis,PNB,3,6,21,6,...,1362.76,medium,android,4G,0.64,0.64,0.04,0.11,NaN,0
3,0c56bbe5-1d3a-403e-b9cf-8bec8271b86b,2025-11-14 21:20:13,8849696532@okbob,8710122691@okaxis,BOB,Axis,5,3,21,4,...,153.70,small,feature_phone,2G,0.69,0.91,0.16,0.01,RB,1
4,9a0738a0-6bab-40ad-9513-f6fc5d2e1204,2025-12-31 11:29:33,4514627048@okhdfc,2814893252@kotak,HDFC,Kotak,1,4,11,2,...,1004.60,medium,feature_phone,2G,0.91,0.79,0.03,0.04,U69,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499995,3f79cf00-8187-40da-85c8-b0caac3633b6,2026-02-27 17:22:28,8947908562@yesbank,7367328769@yesbank,YesBank,YesBank,7,7,17,4,...,2707.92,medium,android,wifi,0.88,0.80,0.09,0.22,NaN,0
499996,9024fffd-cf64-48c3-9897-78f6deb4bb47,2026-03-13 18:27:05,5634403477@oksbi,0044715920@okaxis,SBI,Axis,0,3,18,4,...,1676.40,medium,android,wifi,0.83,0.85,0.33,0.07,NaN,0
499997,aa7110c5-f70f-430d-a939-5a0da523cf26,2025-10-21 07:08:47,7685010565@oksbi,9862781893@oksbi,SBI,SBI,0,0,7,1,...,760.64,small,feature_phone,3G,0.79,0.67,0.07,0.38,NaN,0
499998,261f77b9-0a49-403a-90da-e4daeae9f74d,2025-11-07 23:12:59,1901440521@yesbank,5737197800@okpnb,YesBank,PNB,7,6,23,4,...,737.65,small,android,3G,0.74,0.76,0.19,0.25,Z6,1


In [8]:
df.columns

Index(['transaction_id', 'timestamp', 'sender_vpa', 'receiver_vpa',
       'sender_bank', 'receiver_bank', 'sender_bank_encoded',
       'receiver_bank_encoded', 'hour_of_day', 'day_of_week', 'is_salary_day',
       'is_festival_day', 'amount', 'amount_bucket', 'device_type',
       'network_type', 'sender_bank_health', 'receiver_bank_health',
       'sender_recent_fail_rate', 'receiver_recent_fail_rate',
       'failure_reason_code', 'is_failed'],
      dtype='object')